In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "5"
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets, models
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATASET_PATH = "top_train"
GROUND_PATH = "top_test/top"

NUM_CLASSES = 5
BATCH_SIZE = 16


class CleanImageFolder(datasets.ImageFolder):
    def find_classes(self, directory):
        classes = [
            d.name for d in os.scandir(directory)
            if d.is_dir() and not d.name.startswith(".")
        ]
        classes.sort()
        class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
        return classes, class_to_idx

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.3, 0.3),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor()
])

dataset = CleanImageFolder(DATASET_PATH, transform=transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

ground_dataset = CleanImageFolder(GROUND_PATH, transform=transform)
ground_loader = DataLoader(ground_dataset, batch_size=8, shuffle=True)

print("Classes:", dataset.classes)


model = models.efficientnet_b0(pretrained=True)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.5
)

train_losses = []
val_losses = []
val_accuracies = []

def evaluate(model, loader):
    model.eval()
    correct, total, loss_total = 0, 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            out = model(x)
            loss = criterion(out, y)

            loss_total += loss.item()

            _, preds = torch.max(out, 1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total, loss_total / len(loader)

def train(model, train_loader, val_loader, epochs, optimizer, name=""):

    best_acc = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for x, y in tqdm(train_loader, desc=name):
            x, y = x.to(device), y.to(device)

            out = model(x)
            loss = criterion(out, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        train_loss = total_loss / len(train_loader)
        val_acc, val_loss = evaluate(model, val_loader)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        print(f"\n{name} Epoch {epoch+1}")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Acc: {val_acc:.4f}, Val Loss: {val_loss:.4f}")

        scheduler.step(val_loss)

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_top_model.pth")
            print(" Best model saved")

print("\n Training on SMALL dataset")
train(model, train_loader, val_loader, epochs=8, optimizer=optimizer, name="SMALL")


print("\n Fine-tuning on SMALL GROUND")

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-5)

train(model, ground_loader, val_loader, epochs=5, optimizer=optimizer, name="GROUND")

torch.save(model.state_dict(), "top_final_model.pth")

plt.figure()
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.legend()
plt.title("Loss Curve")
plt.savefig("small_loss.png")

plt.figure()
plt.plot(val_accuracies, label="Val Accuracy")
plt.legend()
plt.title("Accuracy Curve")
plt.savefig("small_accuracy.png")

def plot_confusion(model, loader, classes):
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            out = model(x)
            _, preds = torch.max(out, 1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(y.numpy())

    cm = confusion_matrix(labels_all, preds_all)

    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=classes,
                yticklabels=classes)
    plt.title("Confusion Matrix")
    plt.savefig("small_confusion.png")

    print("\nClassification Report:\n")
    print(classification_report(labels_all, preds_all, target_names=classes))

plot_confusion(model, val_loader, dataset.classes)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "5"

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets, models
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATASET_PATH = "top_train"
NUM_CLASSES = 5
BATCH_SIZE = 32

class CleanImageFolder(datasets.ImageFolder):
    def find_classes(self, directory):
        classes = [
            d.name for d in os.scandir(directory)
            if d.is_dir() and not d.name.startswith(".")
        ]
        classes.sort()
        class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
        return classes, class_to_idx

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.2),
    transforms.RandomAffine(15, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.25)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

dataset = CleanImageFolder(DATASET_PATH, transform=train_transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_ds, val_ds = random_split(dataset, [train_size, val_size])

val_ds.dataset.transform = val_transform

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print("Classes:", dataset.classes)


model = models.efficientnet_b0(weights="IMAGENET1K_V1")

in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, NUM_CLASSES)
)

model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.cuda.amp.GradScaler()

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0

    for x, y in tqdm(loader):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            out = model(x)
            loss = criterion(out, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            out = model(x)
            _, preds = torch.max(out, 1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total


print("\n Stage 1: Training classifier")

for param in model.features.parameters():
    param.requires_grad = False

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

for epoch in range(5):
    loss = train_epoch(model, train_loader, optimizer)
    acc = evaluate(model, val_loader)

    scheduler.step()

    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Val Acc: {acc:.4f}")


print("\n Stage 2: Fine-tuning")

# Unfreeze last layers only
for param in model.features[-3:].parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

best_acc = 0

for epoch in range(15):
    loss = train_epoch(model, train_loader, optimizer)
    acc = evaluate(model, val_loader)

    scheduler.step()

    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Val Acc: {acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "best_model.pth")
        print(" Best model saved")

print(f"\n Best Accuracy: {best_acc:.4f}")